# 🔍 RAG Hands-on Lab — Retrieval-Augmented Generation

What you'll build:
- **Chunk** documents and convert them to **vectors** (embeddings)
- **Store** vectors in **ChromaDB** and search by semantic similarity
- *(Optional)* Connect **Claude API** to generate natural language answers
- Apply the same pipeline to **your own documents**

**Phases 1–3 are completely free** — no API key required.

Run each cell top to bottom with **Shift+Enter**.

---

## Phase 1: Setup
### Cell 1 — Install libraries

In [ ]:
!pip install chromadb sentence-transformers -q
print("✅ Installation complete")

### Cell 2 — Load embedding model

`all-MiniLM-L6-v2`: a lightweight embedding model (~80 MB). Free, runs locally in Colab.

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

# Download model on first run (~80 MB), cached after that
print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model loaded")

# In-memory ChromaDB — no server or account needed
client = chromadb.Client()
collection = client.get_or_create_collection("rag_demo")  # safe to re-run
print("✅ ChromaDB initialized")

---
## Phase 2: Index Documents
### Cell 3 — Prepare sample documents

10 software engineering concepts used as sample documents.

In [ ]:
documents = [
    {
        "id": "git-1",
        "text": "Git is a distributed version control system (DVCS). It tracks changes to source code and allows multiple developers to collaborate simultaneously. Changes are saved with commits, independent work streams are created with branches, and changes are integrated with merge. Used together with remote repositories like GitHub or GitLab."
    },
    {
        "id": "rest-1",
        "text": "REST API is a web service interface based on the HTTP protocol. It uses GET (read), POST (create), PUT (update), and DELETE methods. Resources are represented as URLs, and data is exchanged in JSON format. The stateless property means each request is handled independently."
    },
    {
        "id": "jwt-1",
        "text": "JWT (JSON Web Token) is a token-based method for securely transmitting authentication information. It consists of three parts: Header, Payload, and Signature. The server signs the token with a secret key, and the client includes the token in the Authorization header on every request. Tokens typically expire after 24 hours."
    },
    {
        "id": "docker-1",
        "text": "Docker is a platform for packaging applications as containers. You define the environment in a Dockerfile and build an image. Containers run independently of the host OS, solving the 'it works on my machine' problem. docker-compose manages multiple containers together."
    },
    {
        "id": "cicd-1",
        "text": "CI/CD stands for Continuous Integration and Continuous Deployment. When code is pushed, tests, builds, and deployments run automatically. GitHub Actions, Jenkins, and CircleCI are common tools. CI/CD enables fast feedback and stable deployments."
    },
    {
        "id": "db-index-1",
        "text": "A database index is a data structure that speeds up search queries. B-Tree indexes are the most commonly used. Without an index, the database must scan every row (Full Table Scan). Adding an index improves read speed but slightly reduces write performance. Apply to frequently queried columns."
    },
    {
        "id": "http-1",
        "text": "HTTP status codes indicate the meaning of a server response. Common codes: 200 OK (success), 201 Created, 400 Bad Request (invalid input), 401 Unauthorized (authentication required), 403 Forbidden (no permission), 404 Not Found, 500 Internal Server Error."
    },
    {
        "id": "microservice-1",
        "text": "Microservices architecture breaks an application into small independent services. Each service can be deployed and scaled independently. Services communicate via REST APIs or message queues. More complex than monolithic architecture but offers superior scalability."
    },
    {
        "id": "cache-1",
        "text": "Caching stores frequently used data in fast storage temporarily. Redis and Memcached are popular cache servers. CDNs cache static files on servers worldwide. TTL (Time To Live) sets the cache expiration time. Caching can significantly reduce database load."
    },
    {
        "id": "test-1",
        "text": "Software testing is divided into unit tests, integration tests, and E2E tests. TDD (Test-Driven Development) writes tests before implementation. Jest, JUnit, and PyTest are widely used. Test coverage measures the percentage of code covered by tests."
    }
]

print(f"📄 {len(documents)} sample documents ready\n")
for doc in documents:
    print(f"  [{doc['id']}] {doc['text'][:60]}...")

### Cell 4 — Generate embeddings and store in ChromaDB

Each document is converted to a 384-dimensional vector and stored.

In [ ]:
texts = [doc["text"] for doc in documents]
ids   = [doc["id"]   for doc in documents]

# Text → vector (384 dimensions)
print("Generating embeddings...")
embeddings = model.encode(texts)
print(f"✅ Embeddings done: shape = {embeddings.shape}")
print(f"   → {len(texts)} documents, each a {embeddings.shape[1]}-dimensional vector")

# Store in ChromaDB
collection.add(
    documents=texts,
    embeddings=embeddings.tolist(),
    ids=ids
)
print(f"\n✅ Stored in ChromaDB: {collection.count()} documents")

---
## Phase 3: Search
### Cell 5 — Search with a question

Change the query and observe how the results change.

In [ ]:
# ✏️ Try changing this query!
query = "How long does a token stay valid?"

# Convert query to a vector
query_embedding = model.encode([query]).tolist()

# Find top 3 most similar chunks
results = collection.query(
    query_embeddings=query_embedding,
    n_results=3,
    include=['documents', 'distances']
)

print(f"🔍 Query: {query}\n")
print("Search results (highest similarity first):")
print("-" * 60)

for i, (doc, distance) in enumerate(
    zip(results['documents'][0], results['distances'][0])
):
    similarity = 1 - distance  # cosine distance → similarity
    print(f"\n[#{i+1}] Similarity: {similarity:.3f}")
    print(f"{doc}")

### Cell 6 — Experiment with multiple queries

In [ ]:
if collection.count() == 0:
    print("⚠️  Collection is empty. Run Cells 3 and 4 first, then re-run this cell.")
else:
    test_queries = [
        "How do I track code changes?",
        "How does login authentication work?",
        "What status code means server error?",
        "How to speed up database search?",
        "How to set up automated deployment?",
    ]

    for query in test_queries:
        embedding = model.encode([query]).tolist()
        results = collection.query(
            query_embeddings=embedding,
            n_results=1,
            include=['documents', 'distances', 'ids']
        )
        docs = results['documents'][0]
        distances = results['distances'][0]
        ids = results['ids'][0]

        if not docs:
            print(f"Q: {query}")
            print(f"   → no results found")
            print()
            continue

        similarity = 1 - distances[0]
        print(f"🔍 Query  : {query}")
        print(f"   Score  : {similarity:.2f}  |  ID: {ids[0]}")
        print(f"   Result : {docs[0]}")
        print("-" * 70)
        print()

---
## Phase 4: Add LLM — Optional (API Key Required)
### Cell 7 — Generate a natural language answer with Claude

Skip this cell if you don't have an API key. Phases 1–3 already demonstrate the core of RAG.

In [ ]:
# ✏️ Enter your Anthropic API key (skip this cell if you don't have one)
ANTHROPIC_API_KEY = "sk-ant-..."  # replace with your key

!pip install anthropic -q
import anthropic

llm = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# Search
query = "How does JWT token expiration work?"
embedding = model.encode([query]).tolist()
results = collection.query(
    query_embeddings=embedding,
    n_results=3,
    include=['documents', 'distances']
)

top_similarity = 1 - results['distances'][0][0]
chunks = results['documents'][0]

if top_similarity > 0.5:  # relevant document found
    context = "\n\n".join(chunks)
    prompt = f"Answer based on these documents:\n\n{context}\n\nQuestion: {query}"
    source = "Document-based answer"
else:  # no relevant document
    prompt = f"Question: {query}"
    source = "LLM knowledge-based answer"

response = llm.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=512,
    messages=[{"role": "user", "content": prompt}]
)

print(f"🔍 Query: {query}")
print(f"📊 Top similarity: {top_similarity:.2f} → {source}")
print(f"\n💬 Answer:\n{response.content[0].text}")

---
## Phase 5: Your Own Documents
### Cell 8 — Upload and index your file

In [ ]:
from google.colab import files

print("📁 Upload a text file (.txt)")
uploaded = files.upload()

# New collection for your documents
my_collection = client.create_collection("my_documents")

for filename, content in uploaded.items():
    text = content.decode('utf-8')

    # Split on blank lines (simple paragraph chunking)
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]

    # Remove chunks that are too short
    chunks = [p for p in paragraphs if len(p) > 50]

    print(f"\n📄 {filename}: split into {len(chunks)} chunks")

    # Embed and store
    embeddings = model.encode(chunks).tolist()
    ids = [f"{filename}-{i}" for i in range(len(chunks))]

    my_collection.add(
        documents=chunks,
        embeddings=embeddings,
        ids=ids
    )
    print(f"✅ {len(chunks)} chunks indexed")

### Cell 9 — Search your own documents

In [ ]:
# ✏️ Change this to a question relevant to your document
my_query = "Enter your question here"

embedding = model.encode([my_query]).tolist()
results = my_collection.query(
    query_embeddings=embedding,
    n_results=3,
    include=['documents', 'distances']
)

print(f"🔍 Query: {my_query}\n")
print("Results:")
print("-" * 60)

for i, (doc, distance) in enumerate(
    zip(results['documents'][0], results['distances'][0])
):
    similarity = 1 - distance
    print(f"\n[#{i+1}] Similarity: {similarity:.3f}")
    print(doc)

---
## 🎉 Done!

What you built:

| Step | Technology | What it does |
|------|------------|--------------|
| Chunking | Python | Split documents into small pieces |
| Embedding | sentence-transformers | Text → 384-dim vector |
| Storage | ChromaDB | Store vectors locally |
| Search | cosine similarity | Find semantically similar chunks |
| Answer | Claude API (optional) | Generate natural language response |

**This is what happens inside ChatGPT, Cursor, and GitHub Copilot.**

In production, ChromaDB is replaced by Supabase pgvector or Pinecone for persistence and scale.